In [1]:
import json
import pandas as pd
from pathlib import Path
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from torch.optim import AdamW
from tqdm import tqdm

In [2]:
ROOT_PATH = "/home/stefan/ioai-prep/kits/contest/magic"
TRAIN_DIR = f"{ROOT_PATH}/train"
TEST_DIR = f"{ROOT_PATH}/test"
SUBMISSION_PATH = f"{ROOT_PATH}/submission.csv"
MODEL_PATH = f"{ROOT_PATH}/finetuned_clip"
MODEL_NAME = "openai/clip-vit-large-patch14"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data

In [3]:
def load_train_data(train_dir: str):
    train_records = []
    for meta_path in Path(train_dir).glob("*/metadata.json"):
        with open(meta_path) as f:
            d = json.load(f)

        rec = {
            "id": meta_path.parent.name,
            "image_path": str(meta_path.parent / "image.png"),
            "word_choices": list(dict.fromkeys(d["word_choices"])),
            "correct_words": d["correct_words"],
        }
        train_records.append(rec)

    return train_records


def load_test_data(test_dir: str):
    test_records = []
    for meta_path in Path(test_dir).glob("*/metadata.json"):
        with open(meta_path) as f:
            d = json.load(f)

        rec = {
            "id": meta_path.parent.name,
            "image_path": str(meta_path.parent / "image.png"),
            "word_choices": list(dict.fromkeys(d["word_choices"])),
        }
        test_records.append(rec)

    return test_records

# Training

In [4]:
def train_model(model, train_records, processor, epochs, device, lr):
    model.train()
    optimizer = AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0
        for rec in tqdm(train_records, desc=f"Epoch {epoch+1}/{epochs}"):
            image = Image.open(rec["image_path"]).convert("RGB")
            words = rec["word_choices"]
            correct_words = rec["correct_words"]

            labels = torch.tensor(
                [1.0 if word in correct_words else 0.0 for word in words]
            ).to(device)

            inputs = processor(
                text=words,
                images=image,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=77,
            ).to(device)

            optimizer.zero_grad()
            outputs = model(**inputs)
            logits = outputs.logits_per_image[0]
            loss = F.binary_cross_entropy_with_logits(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_records)
        print(f"Epoch {epoch+1} - Loss: {avg_loss:.4f}")

    return model

In [5]:
train_records = load_train_data(TRAIN_DIR)
print(f"Found {len(train_records)} training samples")

model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

Found 200 training samples


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [6]:
print("Fine-tuning model...")
model = train_model(model, train_records, processor, epochs=5, device=DEVICE, lr=5e-6)

print(f"Saving model to {MODEL_PATH}")
model.save_pretrained(MODEL_PATH)
processor.save_pretrained(MODEL_PATH)

Fine-tuning model...


Epoch 1/5: 100%|██████████| 200/200 [00:33<00:00,  6.06it/s]


Epoch 1 - Loss: 0.5191


Epoch 2/5: 100%|██████████| 200/200 [00:32<00:00,  6.17it/s]


Epoch 2 - Loss: 0.1267


...

Epoch 4/5: 100%|██████████| 200/200 [00:32<00:00,  6.09it/s]


Epoch 4 - Loss: 0.0400


Epoch 5/5: 100%|██████████| 200/200 [00:32<00:00,  6.14it/s]


Epoch 5 - Loss: 0.0145
Saving model to /home/stefan/ioai-prep/kits/contest/magic/finetuned_clip


[]

# Prediction

In [7]:
def predict_top5(record, model, processor, device):
    image = Image.open(record["image_path"]).convert("RGB")
    words = record["word_choices"]

    inputs = processor(
        text=words,
        images=image,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits_per_image[0].cpu().numpy()

    top5_indices = logits.argsort()[::-1][:5]
    return [words[i] for i in top5_indices]

In [8]:
test_records = load_test_data(TEST_DIR)
print(f"Found {len(test_records)} test samples")

model.eval()

print("Generating predictions...")
results = []
for rec in tqdm(test_records):
    top5_words = predict_top5(rec, model, processor, DEVICE)
    results.append(
        {
            "ID": rec["id"],
            "word1": top5_words[0],
            "word2": top5_words[1],
            "word3": top5_words[2],
            "word4": top5_words[3],
            "word5": top5_words[4],
        }
    )

Found 100 test samples
Generating predictions...


  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:06<00:00, 15.54it/s]


In [9]:
submission_df = pd.DataFrame(results)
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Submission saved to: {SUBMISSION_PATH}")
submission_df.head()

Submission saved to: /home/stefan/ioai-prep/kits/contest/magic/submission.csv


,ID,word1,word2,word3,word4,word5
0,00078,tomato,hippo,bell,dryad,jasmine
1,00022,projector,ornament,marker,machine,item
2,00050,aloe vera,onion,watercolor,armadillo,nutcracker
3,00009,peas,tablet,lynx,ornament,cat
4,00013,iris,wreath,vase,scissors,projector
